In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.decomposition import PCA

def locate_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    return current

root = locate_root()
data_dir = root / "data"
metadata_dir = data_dir / "metadata"
processed_dir = data_dir / "processed"
reports_dir = root / "reports"
figures_dir = reports_dir / "figures"
tables_dir = reports_dir / "tables"

for directory in [processed_dir, figures_dir, tables_dir]:
    directory.mkdir(parents=True, exist_ok=True)

standardized_df = pd.read_parquet(processed_dir / "flight_rows_standardized.parquet")
feature_manifest = pd.read_csv(metadata_dir / "feature_manifest.csv")

metadata_columns = ["file_name", "group", "relative_path", "sheet_name", "header_row", "original_row_index", "sequence_index", "time_index", "time_source"]
all_feature_columns = [col for col in standardized_df.columns if col not in metadata_columns]

healthy_df = standardized_df[standardized_df["group"] == "healthy"].reset_index(drop=True)
defective_df = standardized_df[standardized_df["group"] == "defective"].reset_index(drop=True)

pd.DataFrame(
    {
        "metric": ["rows", "healthy_rows", "defective_rows", "all_feature_columns"],
        "value": [len(standardized_df), len(healthy_df), len(defective_df), len(all_feature_columns)],
    }
)

/home/hamza/Documents/aero-res/aircraft_fault_localization/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,metric,value
0,rows,312501
1,healthy_rows,142391
2,defective_rows,170110
3,all_feature_columns,178


In [2]:
selected_manifest = feature_manifest[
    (feature_manifest["healthy_non_null_ratio"].fillna(0) >= 0.8) &
    (feature_manifest["defective_non_null_ratio"].fillna(0) >= 0.4)
].copy()

pca_features = selected_manifest["feature"].astype(str).tolist()

if len(pca_features) == 0:
    raise ValueError("No PCA features satisfied the coverage thresholds.")

healthy_matrix = healthy_df[pca_features].fillna(0).astype(np.float32)

fit_rows = min(len(healthy_matrix), 150000)
healthy_fit = healthy_matrix.sample(fit_rows, random_state=42).reset_index(drop=True) if len(healthy_matrix) > fit_rows else healthy_matrix.copy()

full_pca = PCA(svd_solver="full", random_state=42)
full_pca.fit(healthy_fit)
cumulative_variance = np.cumsum(full_pca.explained_variance_ratio_)
components_for_95 = int(np.searchsorted(cumulative_variance, 0.95) + 1)
n_components = int(min(max(2, components_for_95), min(25, healthy_fit.shape[1], healthy_fit.shape[0])))

pd.DataFrame(
    {
        "metric": ["pca_features", "healthy_fit_rows", "components_for_95", "n_components"],
        "value": [len(pca_features), len(healthy_fit), components_for_95, n_components],
    }
)

,metric,value
0,pca_features,178
1,healthy_fit_rows,142391
2,components_for_95,4
3,n_components,4


In [3]:
def score_pca_frame(frame, pca):
    x = frame.to_numpy(dtype=np.float32, copy=False)
    scores = pca.transform(x)
    reconstructed = pca.inverse_transform(scores)
    residual = x - reconstructed
    spe = np.sum(residual * residual, axis=1).astype(np.float32)
    eigenvalues = np.maximum(pca.explained_variance_, 1e-12)
    t2 = np.sum((scores * scores) / eigenvalues, axis=1).astype(np.float32)
    return scores.astype(np.float32), spe, t2

def build_segments(file_scores, candidate_column):
    file_scores = file_scores.sort_values("sequence_index").reset_index(drop=True)
    candidate = file_scores[candidate_column].astype(bool).to_numpy()
    segments = []
    start = None
    for i, flag in enumerate(candidate):
        if flag and start is None:
            start = i
        if start is not None and ((not flag) or i == len(candidate) - 1):
            end = i if flag and i == len(candidate) - 1 else i - 1
            segment = file_scores.iloc[start:end + 1]
            segments.append(
                {
                    "file_name": segment["file_name"].iloc[0],
                    "group": segment["group"].iloc[0],
                    "segment_start_sequence": int(segment["sequence_index"].min()),
                    "segment_end_sequence": int(segment["sequence_index"].max()),
                    "segment_rows": int(len(segment)),
                    "segment_start_time": float(segment["time_index"].min()),
                    "segment_end_time": float(segment["time_index"].max()),
                    "mean_fault_score": float(segment["pca_fault_score"].mean()),
                    "max_fault_score": float(segment["pca_fault_score"].max()),
                }
            )
            start = None
    return segments

pca = PCA(n_components=n_components, svd_solver="full", random_state=42)
pca.fit(healthy_fit)

_, healthy_spe, healthy_t2 = score_pca_frame(healthy_matrix, pca)

spe_threshold = float(np.quantile(healthy_spe, 0.99))
t2_threshold = float(np.quantile(healthy_t2, 0.99))

healthy_reference = healthy_df[["file_name", "group", "sequence_index", "time_index"]].copy()
healthy_reference["spe"] = healthy_spe
healthy_reference["t2"] = healthy_t2
healthy_reference["spe_ratio"] = healthy_reference["spe"] / spe_threshold
healthy_reference["t2_ratio"] = healthy_reference["t2"] / t2_threshold
healthy_reference["pca_fault_score"] = np.maximum(healthy_reference["spe_ratio"], healthy_reference["t2_ratio"]).astype(np.float32)
healthy_reference["is_candidate"] = (healthy_reference["spe_ratio"] >= 1.0) | (healthy_reference["t2_ratio"] >= 1.0)

pd.DataFrame(
    {
        "metric": ["spe_threshold", "t2_threshold", "healthy_candidate_rate"],
        "value": [spe_threshold, t2_threshold, float(healthy_reference["is_candidate"].mean())],
    }
)

/home/hamza/Documents/aero-res/aircraft_fault_localization/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(


,metric,value
0,spe_threshold,10437.915039
1,t2_threshold,111.125954
2,healthy_candidate_rate,0.018091


In [4]:
row_score_parts = []
flight_summary_rows = []
segment_rows = []

for file_name, file_df in tqdm(standardized_df.groupby("file_name"), total=standardized_df["file_name"].nunique(), desc="Scoring files with PCA"):
    frame = file_df[pca_features].fillna(0).astype(np.float32)
    _, spe, t2 = score_pca_frame(frame, pca)
    result = file_df[["file_name", "group", "sequence_index", "original_row_index", "time_index", "sheet_name"]].copy()
    result["spe"] = spe
    result["t2"] = t2
    result["spe_ratio"] = result["spe"] / spe_threshold
    result["t2_ratio"] = result["t2"] / t2_threshold
    result["pca_fault_score"] = np.maximum(result["spe_ratio"], result["t2_ratio"]).astype(np.float32)
    result["healthy_like"] = (result["spe_ratio"] < 1.0) & (result["t2_ratio"] < 1.0)
    result["fault_candidate"] = (result["spe_ratio"] >= 1.0) | (result["t2_ratio"] >= 1.0)
    result["strong_fault_candidate"] = (result["spe_ratio"] >= 1.5) | (result["t2_ratio"] >= 1.5)
    result["pca_state"] = np.where(result["strong_fault_candidate"], "strong_fault_candidate", np.where(result["fault_candidate"], "fault_candidate", "healthy_like"))
    row_score_parts.append(result)
    flight_summary_rows.append(
        {
            "file_name": file_name,
            "group": result["group"].iloc[0],
            "rows": int(len(result)),
            "healthy_like_rows": int(result["healthy_like"].sum()),
            "fault_candidate_rows": int(result["fault_candidate"].sum()),
            "strong_fault_candidate_rows": int(result["strong_fault_candidate"].sum()),
            "fault_candidate_ratio": float(result["fault_candidate"].mean()),
            "strong_fault_candidate_ratio": float(result["strong_fault_candidate"].mean()),
            "mean_fault_score": float(result["pca_fault_score"].mean()),
            "p95_fault_score": float(result["pca_fault_score"].quantile(0.95)),
            "max_fault_score": float(result["pca_fault_score"].max()),
        }
    )
    segment_rows.extend(build_segments(result, "fault_candidate"))

row_scores = pd.concat(row_score_parts, ignore_index=True).sort_values(
    ["group", "file_name", "sequence_index"],
    ascending=[True, True, True],
).reset_index(drop=True)

flight_summary = pd.DataFrame(flight_summary_rows).sort_values(
    ["group", "fault_candidate_ratio", "max_fault_score"],
    ascending=[True, False, False],
).reset_index(drop=True)

segments_df = pd.DataFrame(segment_rows).sort_values(
    ["group", "max_fault_score", "segment_rows"],
    ascending=[True, False, False],
).reset_index(drop=True) if segment_rows else pd.DataFrame(
    columns=["file_name", "group", "segment_start_sequence", "segment_end_sequence", "segment_rows", "segment_start_time", "segment_end_time", "mean_fault_score", "max_fault_score"]
)

flight_summary.head(20)

Scoring files with PCA:   0%|          | 0/23 [00:00<?, ?it/s]/home/hamza/Documents/aero-res/aircraft_fault_localization/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
Scoring files with PCA:   4%|▍         | 1/23 [00:00<00:10,  2.20it/s]/home/hamza/Documents/aero-res/aircraft_fault_localization/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
Scoring files with PCA:   9%|▊         | 2/23 [00:00<00:05,  3.70it/s]/home/hamza/Documents/aero-res/aircraft_fault_localization/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
Scoring files with PCA:  13%|█▎        | 3/23 [00:00<00:04,  4.68it/s]/home/hamza/Documents/aero-res/a

,file_name,group,rows,healthy_like_rows,fault_candidate_rows,strong_fault_candidate_rows,fault_candidate_ratio,strong_fault_candidate_ratio,mean_fault_score,p95_fault_score,max_fault_score
0,Defect16(01).xlsx,defective,9690,27,9663,731,0.997214,0.075439,1.224429,2.279083,3.286290
1,Defected 5.xlsx,defective,9690,27,9663,731,0.997214,0.075439,1.224429,2.279083,3.286290
2,Defect15(01).xlsx,defective,12431,885,11546,921,0.928807,0.074089,1.171141,2.275640,5.048816
3,Defected 1.xlsx,defective,12431,885,11546,921,0.928807,0.074089,1.171141,2.275640,5.048816
4,Defect15(02).xlsx,defective,4902,1075,3827,3100,0.780702,0.632395,2.389221,5.061831,13.664316
5,Defected 2.xlsx,defective,4902,1075,3827,3100,0.780702,0.632395,2.389221,5.061831,13.664316
6,Defect16(02).xlsx,defective,14537,13134,1403,911,0.096512,0.062668,1.384680,2.177493,71.967888
7,Defected 6.xlsx,defective,14537,13134,1403,911,0.096512,0.062668,1.384680,2.177493,71.967888
8,Defect15(03).xlsx,defective,15689,14350,1339,540,0.085346,0.034419,0.685875,1.137949,5.154141
9,Defected 3.xlsx,defective,15689,14350,1339,540,0.085346,0.034419,0.685875,1.137949,5.154141


In [5]:
explained_variance_df = pd.DataFrame(
    {
        "component": np.arange(1, len(pca.explained_variance_ratio_) + 1, dtype=int),
        "explained_variance_ratio": pca.explained_variance_ratio_,
        "cumulative_explained_variance_ratio": np.cumsum(pca.explained_variance_ratio_),
    }
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(explained_variance_df["component"], explained_variance_df["cumulative_explained_variance_ratio"])
ax.set_xlabel("component")
ax.set_ylabel("cumulative_explained_variance_ratio")
fig.tight_layout()

explained_variance_figure_path = figures_dir / "pca_explained_variance.png"
fig.savefig(explained_variance_figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)

top_defective_files = flight_summary[flight_summary["group"] == "defective"].head(min(6, len(flight_summary[flight_summary["group"] == "defective"])))

if len(top_defective_files):
    fig, axes = plt.subplots(len(top_defective_files), 1, figsize=(16, 3.5 * len(top_defective_files)), sharex=False)
    axes = np.array(axes).reshape(-1)
    for ax, file_name in tqdm(list(zip(axes, top_defective_files["file_name"].tolist())), total=len(top_defective_files), desc="Plotting top defective files"):
        temp = row_scores[row_scores["file_name"] == file_name].sort_values("sequence_index")
        ax.plot(temp["sequence_index"].to_numpy(), temp["pca_fault_score"].to_numpy())
        ax.axhline(1.0)
        ax.set_title(file_name)
        ax.set_xlabel("sequence_index")
        ax.set_ylabel("pca_fault_score")
    fig.tight_layout()
    fault_score_figure_path = figures_dir / "pca_fault_score_top_defective_files.png"
    fig.savefig(fault_score_figure_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
else:
    fault_score_figure_path = figures_dir / "pca_fault_score_top_defective_files.png"

threshold_summary = pd.DataFrame(
    {
        "metric": ["pca_features", "n_components", "spe_threshold", "t2_threshold"],
        "value": [len(pca_features), n_components, spe_threshold, t2_threshold],
    }
)

explained_variance_df.to_csv(tables_dir / "pca_explained_variance.csv", index=False)

str(explained_variance_figure_path), str(fault_score_figure_path)

Plotting top defective files: 100%|██████████| 6/6 [00:00<00:00, 30.15it/s]


('/home/hamza/Documents/aero-res/aircraft_fault_localization/reports/figures/pca_explained_variance.png',
 '/home/hamza/Documents/aero-res/aircraft_fault_localization/reports/figures/pca_fault_score_top_defective_files.png')

In [6]:
row_scores_path = processed_dir / "pca_row_scores.parquet"
flight_summary_path = processed_dir / "pca_flight_summary.csv"
segments_path = processed_dir / "pca_candidate_segments.csv"
healthy_reference_path = processed_dir / "healthy_pca_reference_scores.csv"
threshold_summary_path = processed_dir / "pca_threshold_summary.csv"

row_scores.to_parquet(row_scores_path, index=False)
flight_summary.to_csv(flight_summary_path, index=False)
segments_df.to_csv(segments_path, index=False)
healthy_reference.to_csv(healthy_reference_path, index=False)
threshold_summary.to_csv(threshold_summary_path, index=False)

{
    "pca_row_scores": str(row_scores_path),
    "pca_flight_summary": str(flight_summary_path),
    "pca_candidate_segments": str(segments_path),
    "healthy_pca_reference_scores": str(healthy_reference_path),
    "pca_threshold_summary": str(threshold_summary_path),
    "pca_explained_variance": str(tables_dir / "pca_explained_variance.csv"),
    "pca_explained_variance_figure": str(figures_dir / "pca_explained_variance.png"),
    "pca_fault_score_top_defective_files": str(figures_dir / "pca_fault_score_top_defective_files.png"),
}

{'pca_row_scores': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/processed/pca_row_scores.parquet',
 'pca_flight_summary': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/processed/pca_flight_summary.csv',
 'pca_candidate_segments': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/processed/pca_candidate_segments.csv',
 'healthy_pca_reference_scores': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/processed/healthy_pca_reference_scores.csv',
 'pca_threshold_summary': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/processed/pca_threshold_summary.csv',
 'pca_explained_variance': '/home/hamza/Documents/aero-res/aircraft_fault_localization/reports/tables/pca_explained_variance.csv',
 'pca_explained_variance_figure': '/home/hamza/Documents/aero-res/aircraft_fault_localization/reports/figures/pca_explained_variance.png',
 'pca_fault_score_top_defective_files': '/home/hamza/Documents/aero-res/aircraft_faul